# Trust Analysis Preview

This notebook uses the sample JSON output to confirm that the experiment logs are analysis-ready.

In [ ]:
from collections import defaultdict
from pathlib import Path
import json

sample_path = Path("../data/sample_output.jsonl")
records = [json.loads(line) for line in sample_path.read_text().splitlines() if line.strip()]
records

In [ ]:
summary = defaultdict(lambda: {"total": 0, "accepts": 0, "overrides": 0, "latency_sum": 0})

for record in records:
    bucket = summary[record["condition"]]
    bucket["total"] += 1
    bucket["accepts"] += int(record["decision"] == "accept")
    bucket["overrides"] += int(record["decision"] == "override")
    bucket["latency_sum"] += record["latency_ms"]

analysis_rows = []

for condition, bucket in summary.items():
    total = bucket["total"]
    analysis_rows.append(
        {
            "condition": condition,
            "reliance_rate": round(bucket["accepts"] / total, 3),
            "override_rate": round(bucket["overrides"] / total, 3),
            "mean_latency_ms": round(bucket["latency_sum"] / total, 2),
        }
    )

analysis_rows